# Metrics CERCA

In [1]:
import pandas as pd
import gender_guesser.detector as gender
from df2gspread import gspread2df as g2d

from tqdm import tqdm
tqdm.pandas()

In [2]:
df_whole = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS.csv')

df = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA.csv')
df

,DOI,display_name,author_order,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1002/wps.20971,David R. Williams,58.0,1.362000e+08,US,False,ResearchMar
1,10.1016/j.jaad.2020.10.046,Alexandra Perez Mariño,74.0,NaN,NaN,False,ResearchMar
2,10.1111/all.15679,R. Emuzyte,141.0,1.732121e+08,LT,False,ResearchMar
3,10.1111/all.15679,Thomas Eiwegger,139.0,4.210141e+09,CA,False,ResearchMar
4,10.1111/all.15679,Thomas Eiwegger,139.0,2.801317e+09,CA,False,ResearchMar
...,...,...,...,...,...,...,...
193398,10.1002/clt2.12062,Nikolaos G. Papadopoulos,33.0,2.840731e+07,GB,False,ISGlobal
193399,10.1002/clt2.12062,Nikolaos G. Papadopoulos,33.0,2.840731e+07,GB,False,ResearchMar
193400,10.1016/s2468-2667(21)00065-7,Cătălina Liliana Andrei,33.0,NaN,NaN,False,ISGlobal
193401,10.1007/s00464-024-11109-x,J Guevara-Martínez,33.0,2.800563e+09,ES,False,ResearchMar


In [3]:
df_whole.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA             67
CREAF           933
ICN2            828
ISGlobal        754
ResearchMar    4452
dtype: int64

In [4]:
df[df.CERCA == True].drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA             50
CREAF           825
ICN2            712
ISGlobal        626
ResearchMar    3725
dtype: int64

## Publications Number

In [5]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.9305475504322767


In [6]:
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA             64
CREAF           914
ICN2            773
ISGlobal        727
ResearchMar    4070
dtype: int64

## % led publications

In [7]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8435158501440922


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [8]:
df_led = df[(df.CERCA == True) & (df.author_order == 1)]

df_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA           0.312500
CREAF          0.257112
ICN2           0.344114
ISGlobal       0.337001
ResearchMar    0.309091
dtype: float64

## \% publications with women from the centre as authors

In [9]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8435158501440922


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [11]:
gend = gender.Detector()

df_cerca = df[df.CERCA == True].dropna(subset = 'display_name') # TO DELETE NON FOUND AUTHORS

df_cerca['first_name'] = df_cerca['display_name'].str.split(' ').str[0]
df_cerca['gender'] = df_cerca.first_name.progress_apply(lambda x: gend.get_gender(x))

df_cerca.drop_duplicates('first_name').sort_values('first_name', ascending = False).to_csv('gender_check.csv', index = False)
df_cerca

100%|██████████| 36230/36230 [00:00<00:00, 254347.46it/s]


,DOI,display_name,author_order,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender
88,10.1016/j.drugalcdep.2023.109822,Francina Fonseca,38.0,4.210165e+09,ES,True,ResearchMar,Francina,unknown
116,10.1159/000513538,Josep M. Anto,39.0,4.210148e+09,ES,True,ISGlobal,Josep,male
117,10.1159/000513538,Josep M. Anto,39.0,4.210148e+09,ES,True,ResearchMar,Josep,male
118,10.1159/000513538,Josep M. Anto,39.0,4.210155e+09,ES,True,ISGlobal,Josep,male
119,10.1159/000513538,Josep M. Anto,39.0,4.210155e+09,ES,True,ResearchMar,Josep,male
...,...,...,...,...,...,...,...,...,...
193218,10.3390/jcm10143137,Daniel Guisado‐Alonso,33.0,2.801796e+09,ES,True,ResearchMar,Daniel,male
193300,10.1093/ecco-jcc/jjaa145,Lucía Márquez,33.0,4.210156e+09,ES,True,ResearchMar,Lucía,female
193356,10.1007/s40264-023-01353-w,Daniel Prieto‐Alhambra,33.0,2.801953e+09,NL,True,ResearchMar,Daniel,male
193385,10.1038/s41380-022-01436-7,Carolina Minguillón,33.0,4.210128e+09,ES,True,ResearchMar,Carolina,female


**We manually revise the classifier**

In [12]:
gender_check = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', 'Gender', col_names = True, row_names = False)
df_cerca = df_cerca.merge(gender_check[['first_name', 'gender_check']], on='first_name', how='left')
df_cerca['gender'] = df_cerca.apply(lambda row: row['gender_check'] if row['gender_check'] != '' else row['gender'], axis = 1)
df_cerca

Not all requested scopes were granted by the authorization server, missing scopes https://docs.google.com/feeds, https://spreadsheets.google.com/feeds.


,DOI,display_name,author_order,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender,gender_check
0,10.1016/j.drugalcdep.2023.109822,Francina Fonseca,38.0,4.210165e+09,ES,True,ResearchMar,Francina,female,female
1,10.1159/000513538,Josep M. Anto,39.0,4.210148e+09,ES,True,ISGlobal,Josep,male,
2,10.1159/000513538,Josep M. Anto,39.0,4.210148e+09,ES,True,ResearchMar,Josep,male,
3,10.1159/000513538,Josep M. Anto,39.0,4.210155e+09,ES,True,ISGlobal,Josep,male,
4,10.1159/000513538,Josep M. Anto,39.0,4.210155e+09,ES,True,ResearchMar,Josep,male,
...,...,...,...,...,...,...,...,...,...,...
36225,10.3390/jcm10143137,Daniel Guisado‐Alonso,33.0,2.801796e+09,ES,True,ResearchMar,Daniel,male,
36226,10.1093/ecco-jcc/jjaa145,Lucía Márquez,33.0,4.210156e+09,ES,True,ResearchMar,Lucía,female,
36227,10.1007/s40264-023-01353-w,Daniel Prieto‐Alhambra,33.0,2.801953e+09,NL,True,ResearchMar,Daniel,male,
36228,10.1038/s41380-022-01436-7,Carolina Minguillón,33.0,4.210128e+09,ES,True,ResearchMar,Carolina,female,


In [ ]:
df_fem = df_cerca[df_cerca.gender.isin(['female'])]

df_fem.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA           0.531250
CREAF          0.262582
ICN2           0.460543
ISGlobal       0.594223
ResearchMar    0.632432
dtype: float64

## \% publications led by women from the centre as authors

In [14]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8435158501440922


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [15]:
df_fem_led = df_fem[df_fem.author_order == 1]

df_fem_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA           0.171875
CREAF          0.099562
ICN2           0.116429
ISGlobal       0.226960
ResearchMar    0.158722
dtype: float64

## \% publications in collaboration with other CERCA centres

In [18]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.9305475504322767


In [6]:
df_unique = df[['DOI', 'Center', 'CERCA']].drop_duplicates()
df_cerca = df_unique[df_unique['CERCA'] == True]
for institution in df_cerca.Center.unique():
    dois_this = set(df_cerca.loc[df_cerca.Center == institution, 'DOI'])
    dois_others = set(df_cerca.loc[df_cerca.Center != institution, 'DOI'])
    collaborative_dois = dois_this & dois_others   
    percentage = len(collaborative_dois) / len(dois_this) if dois_this else 0
    print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for ResearchMar is: 2.12%
The percentage of publications in collaboration for ISGlobal is: 12.14%
The percentage of publications in collaboration for BETA is: 8.00%
The percentage of publications in collaboration for CREAF is: 0.48%
The percentage of publications in collaboration for ICN2 is: 0.70%


## \% publications in collaboration with other local institutions

In [19]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.9305475504322767


In [7]:
df_unique = df[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()
df_cerca = df_unique[df_unique['CERCA'] == True]
df_spanish_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] == 'ES')]
for center in df_cerca['Center'].dropna().unique():
    dois_center = set(df_cerca.loc[df_cerca['Center'] == center, 'DOI'])
    dois_spanish_non_cerca = set(df_spanish_non_cerca['DOI'])
    collaborative_dois = dois_center & dois_spanish_non_cerca
    percentage = len(collaborative_dois) / len(dois_center)
    print(f'The percentage of publications analyzed for {center} is: {percentage:.2%}')

The percentage of publications analyzed for ResearchMar is: 79.03%
The percentage of publications analyzed for ISGlobal is: 53.67%
The percentage of publications analyzed for BETA is: 88.00%
The percentage of publications analyzed for CREAF is: 39.88%
The percentage of publications analyzed for ICN2 is: 56.74%


## \% publications in collaboration with other international institutions

In [9]:
df_unique = df[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()

df_cerca = df_unique[df_unique['CERCA'] == True]
df_international_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] != 'ES')]

for center in df_cerca['Center'].dropna().unique():
    dois_center = set(df_cerca.loc[df_cerca['Center'] == center, 'DOI'])
    dois_international = set(df_international_non_cerca['DOI'])

    collaborative_dois = dois_center & dois_international
    
    percentage = len(collaborative_dois) / len(dois_center) if dois_center else 0
    print(f'The percentage of international collaborations for {center} is: {percentage:.2%}')

The percentage of international collaborations for ResearchMar is: 59.73%
The percentage of international collaborations for ISGlobal is: 78.27%
The percentage of international collaborations for BETA is: 58.00%
The percentage of international collaborations for CREAF is: 85.21%
The percentage of international collaborations for ICN2 is: 77.11%
